# 🔬 خط أنابيب ML كامل مع تتبع التجارب (Full ML Pipeline + Experiment Tracking)
### مشروع C1 — مسار علم البيانات (Data Science Track)

---
## 🎯 المشكلة التجارية (Business Problem)
فريق بيبني موديل للتنبؤ بأسعار العقارات، وبيجرّب موديلات وإعدادات كتير. المشكلة:
**"أنهي تجربة كانت الأحسن؟ وبأي إعدادات؟"** — لو مش متسجّل، النتائج بتضيع وما ينفعش تكرّرها.

**الحل:** خط أنابيب (Pipeline) منظّم + **تتبّع التجارب (Experiment Tracking) بـ MLflow** —
كل تجربة بتتسجّل: الإعدادات (params) + المقاييس (metrics) + الموديل نفسه (artifact).

## 📦 ما الذي يثبته المشروع
**Pipeline** (معالجة + موديل) · **Cross-Validation** · **MLflow** (params/metrics/model logging) ·
المقارنة واختيار الأفضل (Model Selection) · استرجاع الموديل المسجّل وتقييمه.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "data_science/c1_mlflow_pipeline"          # مسار المشروع داخل portfolio/
PKGS    = ["mlflow", "xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| Pipeline + ColumnTransformer | Géron — *Hands-On ML* (ch.2) | يمنع تسريب المعالجة ويبسّط الكود |
| Cross-Validation | ISLR (ch.5) / Géron | تقدير أمين لأداء الموديل |
| **تتبّع التجارب (MLflow)** | Huyen — *Designing ML Systems* (ch.6) | تسجيل/مقارنة/تكرار التجارب |
| Model Registry | وثائق MLflow | حفظ الموديل كـ artifact واسترجاعه |
| مقاييس الانحدار (RMSE/MAE/R²) | ISLR (ch.3) | تقييم دقّة التنبؤ |

> 🎯 **بيُستخدم في الواقع:** أي فريق ML جاد بيستخدم تتبّع تجارب (MLflow / Weights & Biases) —
> ده جوهر **MLOps**: تكرارية، مقارنة، وحَوكمة الموديلات.


## 0️⃣ المكتبات والبيانات

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import mlflow, mlflow.sklearn
sns.set_style('whitegrid'); np.random.seed(42)

df = pd.read_csv('data/house_prices_data.csv')
print('Shape:', df.shape, '| MLflow version:', mlflow.__version__)
df.head(3)

Shape: (1500, 8) | MLflow version: 3.13.0


,house_id,square_footage,bedrooms,bathrooms,neighborhood,year_built,garage_spaces,sale_price
0,H10000,2298.0,5,1.0,Rural,1957,2,267876.71
1,H10001,1917.0,3,1.0,Suburbs,1987,2,414178.02
2,H10002,2388.6,4,1.5,Suburbs,2005,1,488565.41


## 1️⃣ استكشاف سريع + هندسة ميزة (Feature Engineering)
نضيف **عمر المنزل (house_age)** بدل سنة البناء — أكثر دلالة للموديل.

In [2]:
df['house_age'] = 2025 - df['year_built']
target = 'sale_price'
print(df[['square_footage','bedrooms','bathrooms','house_age','garage_spaces',target]].describe().round(0).T[['mean','min','max']])
print('\nالحي (neighborhood):', df['neighborhood'].unique())

                    mean      min        max
square_footage    2031.0    600.0     4312.0
bedrooms             4.0      1.0        6.0
bathrooms            2.0      1.0        3.0
house_age           38.0      2.0       75.0
garage_spaces        1.0      0.0        3.0
sale_price      491191.0  82836.0  1039037.0

الحي (neighborhood): ['Rural' 'Suburbs' 'Downtown' 'Uptown']


## 2️⃣ التقسيم وخط المعالجة (Train/Test Split + Preprocessing Pipeline)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

X = df.drop(columns=['house_id', target, 'year_built'])
y = df[target]
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
print('قيم ناقصة في التدريب:', int(X[num].isna().sum().sum()), '→ نعالجها بالـ median داخل الـ Pipeline')

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat)])
print('num:', num, '| cat:', cat, '| train/test:', X_tr.shape[0], '/', X_te.shape[0])

قيم ناقصة في التدريب: 45 → نعالجها بالـ median داخل الـ Pipeline
num: ['square_footage', 'bedrooms', 'bathrooms', 'garage_spaces', 'house_age'] | cat: ['neighborhood'] | train/test: 1200 / 300


## 3️⃣ إعداد MLflow (تتبّع محلي)
نخلّي MLflow يسجّل في قاعدة بيانات محلية `sqlite:///mlflow.db` (مش محتاج سيرفر)، ونعمل
**تجربة (experiment)** باسم محدّد. *(ملاحظة: من MLflow 3 الـ file-store اتحوّل لوضع صيانة،
فالموصى به backend قاعدة بيانات زي sqlite.)*

In [4]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('house-prices-regression')
print('Tracking URI:', mlflow.get_tracking_uri())

2026/06/09 11:44:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/09 11:44:29 INFO mlflow.store.db.utils: Updating database tables


2026/06/09 11:44:30 INFO mlflow.tracking.fluent: Experiment with name 'house-prices-regression' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


## 4️⃣ تدريب ومقارنة الموديلات — كل تجربة في MLflow Run
لكل موديل: نحسب **CV-RMSE**، ندرّب، نقيّم على الاختبار (RMSE/MAE/R²)، ونسجّل **الإعدادات + المقاييس +
الموديل** في run منفصل.

In [5]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

models = {
    'Ridge':        Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'XGBoost':      XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([('pre', pre), ('model', model)])
        cv_rmse = -cross_val_score(pipe, X_tr, y_tr, cv=5,
                                   scoring='neg_root_mean_squared_error').mean()
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_te)
        rmse, mae, r2 = root_mean_squared_error(y_te, pred), mean_absolute_error(y_te, pred), r2_score(y_te, pred)

        mlflow.log_param('model_type', name)
        mlflow.log_params(model.get_params())
        mlflow.log_metrics({'cv_rmse': cv_rmse, 'test_rmse': rmse, 'test_mae': mae, 'test_r2': r2})
        mlflow.sklearn.log_model(pipe, name='model', input_example=X_te.head(2))
        print(f'{name:14} CV-RMSE={cv_rmse:,.0f} | test-RMSE={rmse:,.0f} | R²={r2:.3f}')

2026/06/09 11:44:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\DELL\miniconda3\envs\dsportfolio\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Ridge          CV-RMSE=34,690 | test-RMSE=33,437 | R²=0.962


2026/06/09 11:44:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\DELL\miniconda3\envs\dsportfolio\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


RandomForest   CV-RMSE=28,756 | test-RMSE=25,470 | R²=0.978


2026/06/09 11:44:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\DELL\miniconda3\envs\dsportfolio\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


XGBoost        CV-RMSE=23,836 | test-RMSE=21,015 | R²=0.985


## 5️⃣ مقارنة التجارب واختيار الأفضل (Query Runs)
نسأل MLflow عن كل الـ runs مرتّبة بـ test-RMSE — أقل قيمة = أفضل موديل.

In [6]:
runs = mlflow.search_runs(order_by=['metrics.test_rmse ASC'])
cols = ['tags.mlflow.runName', 'metrics.cv_rmse', 'metrics.test_rmse', 'metrics.test_r2']
print(runs[cols].round(3).to_string(index=False))

best = runs.iloc[0]
best_id = best['run_id']
print(f"\n🏆 الأفضل: {best['tags.mlflow.runName']} | test-RMSE={best['metrics.test_rmse']:,.0f} | run_id={best_id[:8]}")

tags.mlflow.runName  metrics.cv_rmse  metrics.test_rmse  metrics.test_r2
            XGBoost        23835.882          21015.466            0.985
       RandomForest        28755.546          25469.938            0.978
              Ridge        34690.458          33437.118            0.962

🏆 الأفضل: XGBoost | test-RMSE=21,015 | run_id=30e31553


## 6️⃣ استرجاع الموديل المسجّل والتقييم (Load + Residuals)
نحمّل الموديل الأفضل من MLflow (زي ما هيحصل في الإنتاج) ونرسم البواقي (Residuals).

In [7]:
loaded = mlflow.sklearn.load_model(f'runs:/{best_id}/model')
pred = loaded.predict(X_te)
resid = y_te - pred

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(pred, y_te, alpha=0.4);
lims = [y_te.min(), y_te.max()]; ax[0].plot(lims, lims, 'r--')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title('Predicted vs Actual')
ax[1].scatter(pred, resid, alpha=0.4); ax[1].axhline(0, color='r', ls='--')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Residual'); ax[1].set_title('Residuals')
plt.tight_layout(); plt.show()
print('الموديل اتحمّل من MLflow registry بنجاح ✓')

الموديل اتحمّل من MLflow registry بنجاح ✓


C:\Users\DELL\AppData\Local\Temp\ipykernel_42256\1836534313.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **الـ Pipeline** ضمن إن المعالجة (scaling/encoding) تتعلّم من بيانات التدريب بس — لا تسريب.
- **MLflow** سجّل كل تجربة (params + metrics + model) → نقدر نقارن ونكرّر ونرجع لأي تجربة.
- **اختيار الموديل** اتعمل بمقياس موضوعي (test-RMSE) على الـ runs المسجّلة، مش بالحدس.
- **الاسترجاع:** حمّلنا الموديل الأفضل من الـ registry — نفس آلية النشر في الإنتاج.

> 🖥️ **شوف لوحة MLflow:**
> ```bash
> mlflow ui --backend-store-uri sqlite:///mlflow.db      # ثم افتح http://localhost:5000
> ```
> ✅ **اللي اتعلمته:** Pipeline، Cross-Validation، تتبّع التجارب بـ MLflow، المقارنة، واسترجاع الموديل.
